# Reranker Fine-tune PoC — val_009 (clean import edition)

**Hypothesis**: feature-enriched LoRA fine-tuning of Qwen3-Reranker-8B on
Swiss-legal in-domain triplets will deliver a clear R@K lift on val_009 — the
hardest val query.

**This version**: every environment fix and import validation runs in Phase 0.
No mid-notebook pip installs, no `sys.modules` clearing, no Unsloth-imported-
after-transformers traps. If Phase 0 prints `[setup] OK`, every later phase
will load its model and run without env-related failures.

**Pipeline**:
1. Phase 0 — Setup: pip aligns numpy, removes torchcodec, installs core libs,
   imports Unsloth + transformers + peft + torch in the right order, validates.
2. Phase 1 — Load val_009 query + 2000 enriched candidates + 30k triplets.
3. Phase 2 — Define official Qwen3-Reranker prompt template with enriched fields.
4. Phase 3 — Baseline scoring via HF `AutoModelForCausalLM` (no vLLM).
5. Phase 4 — LoRA fine-tune with vanilla transformers+peft (no Unsloth — its
   Triton kernels crash backward on this Torch 2.10 + Blackwell stack).
6. Phase 5 — Merge LoRA + re-score val_009 with the same HF scorer.
7. Phase 6 — Verdict: R@K side-by-side, per-gold rank, F1@K_gold, decision.
8. Phase 7 — Save outputs to Drive.

**Decision criteria**:
- Relative lift on R@K_gold ≥ 50% → CONCLUSIVE YES (proceed to full-scale)
- Lift 20–50%                    → PROMISING (try harder negatives)
- Lift < 20%                     → CONCLUSIVE NO (re-architect)


## Phase 0 — Setup (single cell, all imports validated upfront)

In [ ]:
# =============================================================================
# PHASE 0 — Setup (run from a FRESH KERNEL; do not skip any sub-step)
# =============================================================================
# Critical fixes baked in (each addresses a real bug we hit during dev):
#   0. STALE-KERNEL CHECK at top: if transformers 5.x is already loaded in
#      memory, pip cannot dislodge it. Fail loud with restart instructions.
#   1. Strict pin transformers==4.56.2 (LAST 4.x release with full Qwen3
#      support).
#   2. Align numpy to in-memory version (avoid pip-meta vs runtime clash).
#   3. Uninstall torchcodec + bitsandbytes (their broken imports + cu13 libs
#      poison the python env even if we don't use them).
#   4. NO Unsloth: its Triton RMSNorm/attention kernels crash backward()
#      with a MulBackward0 in-place op error on Torch 2.10 + Blackwell.
#      We use vanilla transformers + peft + SDPA in Phase 4 instead
#      (~2x slower but bulletproof).
#   5. Fix LD_LIBRARY_PATH for the pip-installed CUDA13 runtime libs.
#   6. Post-install assertion: transformers MUST be 4.x or fail loud.

import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path

# ---- CUDA allocator: reduce fragmentation OOM ----------------------------
# Must be set BEFORE the first CUDA op (model load) so the allocator picks
# it up at init. Phase 0 only does light CUDA queries (is_available,
# get_device_name) so we're well within the safe window.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Step 0: STALE-KERNEL CHECK -------------------------------------------
# If a previous Phase 0 run in this kernel already loaded transformers 5.x,
# pip install can downgrade the on-disk version but the cached module stays
# in memory. The Qwen3 in-place op bug will keep triggering. RESTART is the
# only fix.
if "transformers" in sys.modules:
    _v = getattr(sys.modules["transformers"], "__version__", "?")
    if not _v.startswith("4."):
        raise RuntimeError(
            f"\n\n>>> STALE KERNEL: transformers {_v} is already loaded <<<\n"
            f">>> This notebook requires transformers 4.x (5.x has Qwen3 autograd bugs). <<<\n"
            f">>> Colab menu: Runtime -> Restart runtime, then re-run Phase 0. <<<\n"
        )

IS_COLAB = "google.colab" in sys.modules
print(f"[setup] Colab: {IS_COLAB}")

# ---- Step 1: mount Drive --------------------------------------------------
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

def _pip(*args, check=True):
    """pip wrapper using the running python so PATH lookups can't go wrong."""
    return subprocess.run(
        [sys.executable, "-m", "pip", *args],
        check=check, capture_output=True, text=True,
    )

if IS_COLAB:
    # ---- Step 2: align numpy ------------------------------------------
    import numpy as _np
    loaded_np = _np.__version__
    print(f"[setup] numpy loaded in memory: {loaded_np}  --> aligning pip metadata")
    r = _pip("install", "-q", f"numpy=={loaded_np}", check=False)
    if r.returncode != 0:
        print(f"  warn: numpy alignment returned {r.returncode}: {r.stderr[-300:]}")

    # ---- Step 3: uninstall environment-poisoning packages -------------
    # torchcodec:    RuntimeError on missing FFmpeg pollutes the python env.
    # bitsandbytes:  cu13 libs don't load; not needed for bf16 LoRA.
    # torchao:       peft's is_torchao_available() raises ImportError on
    #                old versions (Colab ships 0.10.0, peft requires >=0.16.0)
    #                instead of returning False. Crashes get_peft_model().
    print(f"[setup] removing torchcodec + bitsandbytes + torchao (defensive cleanup)")
    _pip("uninstall", "-y", "torchcodec",   check=False)
    _pip("uninstall", "-y", "bitsandbytes", check=False)
    _pip("uninstall", "-y", "torchao",      check=False)

    # ---- Step 4: install with STRICT transformers pin -----------------
    # NOTE: Unsloth is intentionally NOT installed — its custom Triton kernels
    # crash backward() on Torch 2.10 + Blackwell with a MulBackward0 version
    # mismatch. We use vanilla transformers + peft + SDPA in Phase 4 instead.
    TRANSFORMERS_PIN = "transformers==4.56.2"   # last 4.x; stable Qwen3 era
    print(f"[setup] installing {TRANSFORMERS_PIN} + peft + accelerate (~2 min) ...")
    print(f"        (strict pin: transformers 5.x has Qwen3 in-place op bug)")
    print(f"        (no Unsloth: its Triton kernels crash on Blackwell + Torch 2.10)")
    t0 = time.time()
    r = _pip(
        "install", "-q", "-U",
        TRANSFORMERS_PIN,
        "peft>=0.13.0", "accelerate>=1.0.0",
        "datasets>=3.0.0",
        "pandas>=2.2.0", "pyarrow>=16.0.0", "tqdm",
        check=False,
    )
    print(f"  done in {(time.time()-t0)/60:.1f} min  (returncode={r.returncode})")
    if r.returncode != 0:
        print("=== install stdout (tail) ==="); print(r.stdout[-1500:])
        print("=== install stderr (tail) ==="); print(r.stderr[-1500:])
        raise RuntimeError("core package install failed — see output above")

    # ---- Step 5: verify on-disk transformers version ------------------
    show = _pip("show", "transformers", check=False)
    on_disk = None
    for line in show.stdout.split("\n"):
        if line.startswith("Version:"):
            on_disk = line.split(":", 1)[1].strip(); break
    print(f"[setup] transformers on disk: {on_disk}")
    if on_disk is None or not on_disk.startswith("4."):
        raise RuntimeError(
            f"transformers on disk is {on_disk}, expected 4.x — pip pin failed"
        )

    # ---- Step 6: LD_LIBRARY_PATH for CUDA13 ---------------------------
    CU13_LIB = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"
    if os.path.isdir(CU13_LIB) and CU13_LIB not in os.environ.get("LD_LIBRARY_PATH", ""):
        os.environ["LD_LIBRARY_PATH"] = CU13_LIB + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        print(f"[setup] LD_LIBRARY_PATH += {CU13_LIB}")

    # Blackwell SM 12.0 compile target (RTX 6000 Blackwell / RTX 5090)
    os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"

# ---- Step 7: validated imports -----------------------------------------
print(f"\n[setup] validating imports ...")

import torch
assert torch.cuda.is_available(), "[setup] FATAL: torch reports no CUDA"
print(f"  torch:        {torch.__version__}  (cuda {torch.version.cuda})")
print(f"  GPU:          {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)")

import transformers
print(f"  transformers: {transformers.__version__}")
# CRITICAL ASSERT — must be 4.x. If this fails the kernel is stale.
assert transformers.__version__.startswith("4."), (
    f"\n\nFATAL: transformers must be 4.x for Qwen3 LoRA (got {transformers.__version__}).\n"
    f"This kernel is stale (a previous import cached 5.x in memory).\n"
    f"Fix: Runtime -> Restart runtime, then re-run Phase 0 from the top.\n"
)
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
print(f"  transformers loaders: OK")

import peft
print(f"  peft:         {peft.__version__}")
from peft import PeftModel
print(f"  peft.PeftModel: OK")

import accelerate
print(f"  accelerate:   {accelerate.__version__}")

import pandas as pd
import numpy as np
print(f"  pandas:       {pd.__version__}")
print(f"  numpy:        {np.__version__}")

# ---- Step 8: sanity-check libcudart.so.13 --------------------------------
import ctypes
try:
    ctypes.CDLL("libcudart.so.13")
    print(f"  libcudart.so.13: loads OK")
except OSError as e:
    print(f"  libcudart.so.13: WARN — {e}  (non-fatal unless a subprocess needs it)")

print(f"\n[setup] OK — environment ready for all downstream phases")
print(f"        transformers={transformers.__version__}  (Qwen3 4.x stable, autograd-clean)")


## Phase 1 — Paths + load val_009 query + enriched candidates

In [ ]:
# =============================================================================
# PHASE 1 — Paths, config, and load val_009 query + 2000 enriched candidates
# =============================================================================
# No imports here — Phase 0 already imported pd, np, json, etc.

DRIVE_ROOT     = Path("/content/drive/MyDrive/swiss_law")
VAL_CSV        = DRIVE_ROOT / "data"     / "val.csv"
VAL_ASPECTS    = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"
TRAIN_PARQ     = DRIVE_ROOT / "research" / "reranker_train" / "reranker_train_enriched_30k.parquet"
VAL009_PARQ    = DRIVE_ROOT / "research" / "reranker_train" / "val009_candidates_enriched.parquet"

OUT_DIR        = DRIVE_ROOT / "research" / "reranker_finetune_poc_val009"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR       = OUT_DIR / "lora_adapter"
LORA_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR     = OUT_DIR / "qwen3_reranker_8b_swiss_legal_merged"

QID            = "val_009"
MODEL_NAME     = "Qwen/Qwen3-Reranker-8B"

# Two separate caps — never silently overridden later
MAX_LEN_SCORE  = 1536       # vLLM-style scoring (wider, accommodates full enriched candidate)
MAX_LEN_TRAIN  = 1024       # Unsloth training (narrower, keep VRAM bounded)
LR             = 1e-4
BATCH_SIZE     = 8
GRAD_ACCUM     = 2          # effective batch 16
EPOCHS         = 1
LORA_RANK      = 16
LORA_ALPHA     = 32

SCORE_INSTRUCTION = (
    "Given a Swiss legal question, decide whether a competent Swiss lawyer "
    "writing a legal opinion on the Query would cite the Document. Consider "
    "matching legal codes, legal areas, legal topics, and key concepts."
)

print(f"Verifying inputs:")
for p in [VAL_CSV, VAL_ASPECTS, TRAIN_PARQ, VAL009_PARQ]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

# --- val_009 question text ------------------------------------------------
val_df = pd.read_csv(VAL_CSV)
query_text = str(val_df[val_df.query_id == QID].iloc[0]["query"])
print(f"\n=== val_009 question ({len(query_text)} chars) ===\n{query_text[:600]}...")

# --- val_009 derived metadata (aspects -> codes/areas/topics/concepts) ----
asp_df = pd.read_parquet(VAL_ASPECTS)
v009_row = asp_df[asp_df.query_id == QID].iloc[0]
aspects = list(v009_row["aspects"])
print(f"\n=== val_009 aspects ({len(aspects)}) ===")
for a in aspects:
    print(f"  {a.get('id')}: {a.get('label')}  (weight={float(a.get('weight',0)):.2f})")

def _to_list(v):
    if v is None: return []
    try: return list(v)
    except TypeError: return []

expected_codes = _to_list(v009_row.get("expected_codes"))
target_topics  = [a.get("label","") for a in aspects if a.get("label")]
target_concepts = []
for a in aspects:
    for c in _to_list(a.get("concepts_en")):
        target_concepts.append(c)
target_concepts = list(dict.fromkeys(target_concepts))[:8]

target_areas = []
area_keywords = {
    "maintenance": "civil law", "child": "civil law", "marriage": "civil law",
    "divorce": "civil law", "property": "civil law",
    "enforcement": "civil law / debt enforcement",
    "appeal": "federal supreme court procedure",
    "detention": "criminal procedure", "criminal": "criminal procedure",
    "tax": "tax law", "labor": "labor law", "employment": "labor law",
    "capitalization": "civil law",
}
for a in aspects:
    lbl = (a.get("label","") or "").lower()
    for k, area in area_keywords.items():
        if k in lbl:
            target_areas.append(area); break
target_areas = list(dict.fromkeys(target_areas))

query_meta = {
    "target_codes":    ", ".join(expected_codes) if expected_codes else "(any)",
    "target_areas":    ", ".join(target_areas)  if target_areas  else "(any)",
    "target_topics":   "; ".join(target_topics)[:300],
    "target_concepts": ", ".join(target_concepts)[:300],
}
print("\n=== val_009 derived query metadata ===")
for k, v in query_meta.items():
    print(f"  {k}: {v}")

# --- candidates and training triplets -------------------------------------
val_cands = pd.read_parquet(VAL009_PARQ)
print(f"\nLoaded {len(val_cands)} val_009 candidates  "
      f"(gold_in_topk = {int(val_cands.is_gold.sum())} of {len(val_cands)})")

train_df = pd.read_parquet(TRAIN_PARQ)
print(f"Loaded {len(train_df):,} training triplets")


## Phase 2 — Define Qwen3-Reranker prompt format (official template + enriched fields)

The official Qwen3-Reranker chat template (system prefix + `<Instruct>`/`<Query>`/
`<Document>` + `<think>` suffix) is used verbatim. Enriched lawyer-style metadata
is embedded inside the `<Query>` and `<Document>` bodies — this preserves the
base model's reranker prior while exposing every feature the lawyer's mental
checklist relies on.


In [ ]:
# =============================================================================
# PHASE 2 — Qwen3-Reranker prompt format (official template + enriched fields)
# =============================================================================
# Pure function definitions; no model loading.

QWEN3_RR_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and "
    "the Instruct provided. Note that the answer can only be \"yes\" or \"no\"."
    "<|im_end|>\n<|im_start|>user\n"
)
QWEN3_RR_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

def render_query_content(text, target_codes, target_areas, target_topics, target_concepts):
    """Body of <Query>: — keeps the original question plus the structured aspects."""
    return (
        f"{text[:400]}\n"
        f"target_codes: {target_codes} | target_areas: {target_areas} | "
        f"target_topics: {target_topics} | target_concepts: {target_concepts}"
    )

def render_document_content(citation, code, area, role, title, topic, concepts,
                            chapter_neighbors, popularity_bucket, text):
    """Body of <Document>: — leads with the citation tuple, then text."""
    return (
        f"citation: {citation} | code: {code} | area: {area} | role: {role}\n"
        f"title: {title[:120]}\n"
        f"topic: {topic[:120]}\n"
        f"concepts: {concepts[:200]}\n"
        f"chapter_neighbors: {chapter_neighbors[:200]}\n"
        f"popularity: {popularity_bucket}\n"
        f"text: {text[:700]}"
    )

def render_query_for_score(query_text, query_meta, instruction=SCORE_INSTRUCTION):
    q_body = render_query_content(
        query_text,
        query_meta["target_codes"], query_meta["target_areas"],
        query_meta["target_topics"], query_meta["target_concepts"],
    )
    return f"{QWEN3_RR_PREFIX}<Instruct>: {instruction}\n<Query>: {q_body}\n"

def render_document_for_score(candidate_fields):
    d_body = render_document_content(**candidate_fields)
    return f"<Document>: {d_body}{QWEN3_RR_SUFFIX}"

def format_full_prompt_for_training(query_text, query_meta, candidate_fields,
                                    instruction=SCORE_INSTRUCTION):
    return (
        render_query_for_score(query_text, query_meta, instruction)
        + render_document_for_score(candidate_fields)
    )

def bucketize(n):
    n = int(n)
    if n >= 1000: return "very_high"
    if n >= 100:  return "high"
    if n >= 10:   return "medium"
    return "low"

# --- sanity render on a known-gold candidate -------------------------------
_sample_row = val_cands[val_cands.is_gold].iloc[0]
_sample_fields = {
    "citation":          _sample_row["citation"],
    "code":              _sample_row["code"],
    "area":              _sample_row["area"] or "",
    "role":              _sample_row["role"] or "(unknown)",
    "title":             _sample_row.get("title", "") or "",
    "topic":             _sample_row.get("topic", "") or "",
    "concepts":          _sample_row.get("concepts", "") or "",
    "chapter_neighbors": _sample_row.get("chapter_neighbors", "") or "(none)",
    "popularity_bucket": _sample_row["popularity_bucket"],
    "text":              _sample_row["text"],
}
_sample_prompt = format_full_prompt_for_training(query_text, query_meta, _sample_fields)
print(f"=== Sample full prompt for a GOLD val_009 candidate ({_sample_row['citation']}) ===")
print(f"Total length: {len(_sample_prompt)} chars")
print("-" * 80)
print(_sample_prompt)


## Phase 3 — Baseline scoring of val_009 candidates with off-the-shelf Qwen3-Reranker-8B

Uses HF transformers (`AutoModelForCausalLM`) for scoring rather than vLLM
because vLLM 0.21 dropped the `Qwen3ForSequenceClassification` conversion path
we relied on. We get the same yes/no probability by softmax over the last-token
logits at the (yes_id, no_id) positions.


In [ ]:
# =============================================================================
# PHASE 3 — Baseline scoring via HF transformers
# =============================================================================
# Loads Qwen3-Reranker-8B in bf16 on CUDA, builds 2000 prompts, batches through
# the model, extracts P(yes) from last-token (yes,no) logits.

print(f"Loading {MODEL_NAME} for HF-transformers scoring ...")
qwen3_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if qwen3_tok.pad_token is None:
    qwen3_tok.pad_token = qwen3_tok.eos_token

scorer_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True,
).eval()

yes_id = qwen3_tok.convert_tokens_to_ids("yes")
no_id  = qwen3_tok.convert_tokens_to_ids("no")
print(f"  yes_id={yes_id}, no_id={no_id}")
print(f"  GPU mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def build_prompt(row):
    cand_fields = {
        "citation":          row.citation,
        "code":              row.code,
        "area":              row.area or "",
        "role":              row.role or "(unknown)",
        "title":             row.title or "",
        "topic":             row.topic or "",
        "concepts":          row.concepts or "",
        "chapter_neighbors": row.chapter_neighbors or "(none)",
        "popularity_bucket": row.popularity_bucket,
        "text":              row.text or "",
    }
    return format_full_prompt_for_training(query_text, query_meta, cand_fields)

print(f"\nBuilding {len(val_cands)} prompts ...")
prompts = [build_prompt(r) for r in val_cands.itertuples()]
print(f"  mean prompt length: {sum(len(p) for p in prompts)//len(prompts)} chars")

BATCH_SIZE_SCORE = 8

def score_with_hf(model, tokenizer, prompts, batch_size=BATCH_SIZE_SCORE, max_len=MAX_LEN_SCORE):
    """Yes/no probability via softmax over last-token (no, yes) logits.

    Memory-conscious: uses logits_to_keep=1 so the model computes lm_head
    only at the LAST token (where we extract yes/no), not over the full
    sequence. Cuts lm_head output from [B,S,V] to [B,1,V] — ~1000x smaller
    when S=1024 and V=151k.

    padding_side='left' on the tokenizer means index [-1] is the real last
    token of every sequence in the batch.
    """
    scores = []
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_len).to("cuda")
            out = model(**inputs, logits_to_keep=1)
            logits = out.logits[:, -1, :]   # [B, V]
            yes_logits = logits[:, yes_id].float()
            no_logits  = logits[:, no_id].float()
            pair = torch.stack([no_logits, yes_logits], dim=1)
            probs = torch.nn.functional.softmax(pair, dim=1)[:, 1]
            scores.extend(probs.cpu().tolist())
            del out, logits, yes_logits, no_logits, pair, probs, inputs
            if (i // batch_size) % 20 == 0:
                elapsed = (time.time()-t0)/60
                eta = elapsed / max(1, i+batch_size) * (len(prompts) - i - batch_size)
                print(f"  scored {i+len(batch):>4}/{len(prompts)}  "
                      f"elapsed={elapsed:.1f}min  eta={eta:.1f}min")
    return scores

print(f"\nBaseline scoring (HF transformers, batch={BATCH_SIZE_SCORE}, logits_to_keep=1) ...")
t0 = time.time()
baseline_scores = score_with_hf(scorer_model, qwen3_tok, prompts)
print(f"Done in {(time.time()-t0)/60:.1f} min  ({len(baseline_scores)} scores)")
val_cands["baseline_score"] = baseline_scores

gold_mean = val_cands[val_cands.is_gold].baseline_score.mean()
non_mean  = val_cands[~val_cands.is_gold].baseline_score.mean()
print(f"\nBaseline gold mean:     {gold_mean:.4f}  (n={int(val_cands.is_gold.sum())})")
print(f"Baseline non-gold mean: {non_mean:.4f}  (n={int((~val_cands.is_gold).sum())})")
print(f"Separation:             {gold_mean-non_mean:+.4f}")

# Persist baseline scores to disk — survives kernel restart / OOM in Phase 4
val_cands.to_parquet(OUT_DIR / "val009_baseline_scores.parquet", index=False)
print(f"\n[checkpoint] baseline scores saved to {OUT_DIR}/val009_baseline_scores.parquet")
print(f"            (if kernel dies later, reload via pd.read_parquet to skip Phase 3)")

# AGGRESSIVE cleanup before Phase 4 — don't leak GPU mem across phases.
# scorer_model is ~15 GB; we want it fully released so Phase 4 starts clean.
del scorer_model
del baseline_scores
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"\nGPU mem after Phase 3 cleanup:")
print(f"  allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"  reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")

# Cache prompts for Phase 5 to re-score with the fine-tuned model on the same set
_baseline_prompts = prompts


## Phase 4 — LoRA fine-tune on 30k enriched triplets (vanilla HF + peft, bf16)

In [ ]:
# =============================================================================
# PHASE 4 — LoRA fine-tune (memory-managed, checkpoint-aware)
# =============================================================================
# All bugs we've hit before this version are addressed:
#
#   * MulBackward0 in-place crash — fixed by NOT using Unsloth's Triton kernels.
#   * OOM on logits.float() at step ~3100 — fixed by `logits_to_keep=1` so the
#     model computes lm_head ONLY at the last token. Old logits tensor was
#     [B, S, V] = [8, 1024, 151k] = 2.4 GB bf16 → 5 GB fp32 spike. New logits
#     tensor is [B, 1, V] = 2.4 MB. Plus we skip HF's ForCausalLMLoss entirely
#     and compute CE manually only on the slim slice.
#   * Phase 3 residual GPU mem leaking into Phase 4 — fixed by aggressive
#     cleanup at end of Phase 3 + assert at start of Phase 4.
#   * Lost progress on crash — fixed by saving the LoRA adapter every
#     CKPT_EVERY steps so the worst case is losing < CKPT_EVERY steps.
#   * Memory fragmentation — fixed by PYTORCH_CUDA_ALLOC_CONF=
#     expandable_segments:True (set in Phase 0).
#
# Conservative training config: BATCH=4, GRAD_ACCUM=4 (effective batch 16,
# same as before) — halves activation memory per step. Peak GPU expected
# ~40-50 GB out of 95 GB. Wall-clock ~120-150 min for 1 epoch.

# --- Pre-flight check: transformers must be 4.x ---------------------------
import transformers
assert transformers.__version__.startswith("4."), (
    f"\n\nFATAL: transformers {transformers.__version__} loaded — Phase 4 needs 4.x.\n"
    f"This means the kernel is stale. Runtime -> Restart runtime, then run from Phase 0.\n"
)
print(f"[pre-flight] transformers={transformers.__version__}")

from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# --- Aggressive pre-flight memory cleanup (Phase 3 may have leaked) -------
for _v in ("scorer_model", "model", "merged", "base", "optim", "sched", "dl", "ds"):
    if _v in globals():
        del globals()[_v]
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
_alloc_pre = torch.cuda.memory_allocated() / 1024**3
print(f"\nGPU mem before Phase 4: allocated={_alloc_pre:.2f} GB, "
      f"reserved={torch.cuda.memory_reserved()/1024**3:.2f} GB")
if _alloc_pre > 2.0:
    print(f"  WARN: {_alloc_pre:.1f} GB still allocated from a previous phase. "
          f"  Phase 4 expects < 2 GB. If you hit OOM, restart kernel and "
          f"  reload baseline_scores from {OUT_DIR}/val009_baseline_scores.parquet")

# --- Conservative batch config for safety -------------------------------
# Halve batch size (was 8 → OOM at step 3100). Same effective batch (16)
# by doubling grad accum. Trades 2x wall-clock for ~2x activation headroom.
BATCH_SIZE_TRAIN = 4
GRAD_ACCUM_TRAIN = 4
CKPT_EVERY       = 500   # save adapter every N optimizer steps

print(f"Config: BATCH={BATCH_SIZE_TRAIN}, GRAD_ACCUM={GRAD_ACCUM_TRAIN} "
      f"(eff={BATCH_SIZE_TRAIN*GRAD_ACCUM_TRAIN}), MAX_LEN_TRAIN={MAX_LEN_TRAIN}, "
      f"CKPT_EVERY={CKPT_EVERY}")

# --- Load tokenizer + base model (vanilla HF, SDPA, no Unsloth) ----------
print(f"\nLoading {MODEL_NAME} (bf16, SDPA, no Unsloth) ...")
t0 = time.time()
qwen3_tok = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left",
)
if qwen3_tok.pad_token is None:
    qwen3_tok.pad_token = qwen3_tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
    attn_implementation="sdpa",
)
print(f"  loaded in {time.time()-t0:.1f}s  ({torch.cuda.memory_allocated()/1024**3:.1f} GB)")

yes_id = qwen3_tok("yes", add_special_tokens=False).input_ids[0]
no_id  = qwen3_tok("no",  add_special_tokens=False).input_ids[0]
print(f"  yes_id={yes_id}, no_id={no_id}")

# --- HF gradient checkpointing (use_reentrant=False is autograd-safe) ----
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# --- Apply LoRA via peft -------------------------------------------------
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU mem after LoRA setup: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# --- Dataset: Approach B (no yes/no append; target is a separate field) --
# Why: with logits_to_keep=1, the model returns only the LAST position's
# logits. That logit predicts the NEXT token. So the cleanest design is to
# have the prompt end where we want the model to start generating, and
# train it to emit yes/no as the next token. Matches inference time exactly.
class EnrichedSFTDataset(Dataset):
    """Prompt (no answer token appended) + separate yes/no target field."""

    def __init__(self, df, tok, max_len, yes_id, no_id):
        self.tok = tok
        self.max_len = max_len
        self.yes_id = int(yes_id)
        self.no_id  = int(no_id)
        self.examples = []
        for r in df.itertuples():
            q_meta = {
                "target_codes":    str(r.target_codes or "(any)"),
                "target_areas":    str(r.target_areas or "(any)"),
                "target_topics":   str(r.target_topics or "")[:300],
                "target_concepts": str(r.target_concepts or "")[:300],
            }
            for side, is_pos in [("pos", True), ("neg", False)]:
                cand_fields = {
                    "citation":          str(getattr(r, f"{side}_citation")),
                    "code":              str(getattr(r, f"{side}_code")),
                    "area":              str(getattr(r, f"{side}_area") or ""),
                    "role":              str(getattr(r, f"{side}_role") or "(unknown)"),
                    "title":             str(getattr(r, f"{side}_title") or ""),
                    "topic":             str(getattr(r, f"{side}_topic") or ""),
                    "concepts":          str(getattr(r, f"{side}_concepts") or ""),
                    "chapter_neighbors": str(getattr(r, f"{side}_chapter_neighbors") or "(none)"),
                    "popularity_bucket": bucketize(getattr(r, f"{side}_popularity")),
                    "text":              str(getattr(r, f"{side}_text") or "")[:600],
                }
                self.examples.append((r.query, q_meta, cand_fields, is_pos))

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        q, q_meta, cand, is_pos = self.examples[idx]
        target_id = self.yes_id if is_pos else self.no_id
        prompt = format_full_prompt_for_training(q, q_meta, cand)
        enc = self.tok(prompt, return_tensors="pt", add_special_tokens=False,
                       max_length=self.max_len, truncation=True)
        return {"input_ids": enc.input_ids[0], "target": target_id}

def collate(batch):
    """Left-pad to the longest sequence in this batch; emit target tensor."""
    max_len_b = max(b["input_ids"].size(0) for b in batch)
    pad_id = qwen3_tok.pad_token_id or qwen3_tok.eos_token_id
    input_ids = torch.full((len(batch), max_len_b), pad_id, dtype=torch.long)
    attn      = torch.zeros((len(batch), max_len_b), dtype=torch.long)
    target    = torch.zeros(len(batch), dtype=torch.long)
    for i, b in enumerate(batch):
        L = b["input_ids"].size(0)
        input_ids[i, -L:] = b["input_ids"]
        attn[i,      -L:] = 1
        target[i] = b["target"]
    return {"input_ids": input_ids, "attention_mask": attn, "target": target}

ds = EnrichedSFTDataset(train_df, qwen3_tok, MAX_LEN_TRAIN, yes_id, no_id)
dl = DataLoader(ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True, collate_fn=collate,
                num_workers=2, pin_memory=True)
print(f"\nDataset: {len(ds):,} examples, {len(dl):,} batches per epoch")

# Sanity-check: target is always yes_id or no_id (matches is_pos flag)
for i in range(min(6, len(ds))):
    item = ds[i]
    expected = yes_id if ds.examples[i][3] else no_id
    assert int(item["target"]) == expected, (
        f"Example {i}: target {item['target']} != expected {expected}"
    )
print("  target sanity-check passed (6 examples).")

# --- Optimizer + LR schedule ---------------------------------------------
total_steps = len(dl) * EPOCHS // GRAD_ACCUM_TRAIN
optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = get_linear_schedule_with_warmup(
    optim, num_warmup_steps=int(0.03*total_steps), num_training_steps=total_steps,
)

print(f"\nTraining: {total_steps:,} optimizer steps over {EPOCHS} epoch(s)")
print(f"  Effective batch={BATCH_SIZE_TRAIN*GRAD_ACCUM_TRAIN}, LR={LR}, max_len={MAX_LEN_TRAIN}")
print(f"  Save checkpoint every {CKPT_EVERY} steps -> {LORA_DIR}")
print(f"  GPU mem at training start: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def save_checkpoint(step_n):
    """Save LoRA adapter + tokenizer to LORA_DIR (overwrites)."""
    model.save_pretrained(str(LORA_DIR))
    qwen3_tok.save_pretrained(str(LORA_DIR))
    # Tiny progress marker so we know which step it was from
    with open(LORA_DIR / "_last_step.txt", "w") as _f:
        _f.write(str(step_n))

# --- Training loop -------------------------------------------------------
t_start = time.time()
running_loss = 0.0
step = 0
optim.zero_grad()
for epoch in range(EPOCHS):
    for i, batch in enumerate(dl):
        input_ids = batch["input_ids"].cuda(non_blocking=True)
        attn_mask = batch["attention_mask"].cuda(non_blocking=True)
        target    = batch["target"].cuda(non_blocking=True)

        # logits_to_keep=1: only run lm_head on the LAST position.
        # Output logits shape: [B, 1, V] instead of [B, S, V].
        out = model(input_ids=input_ids, attention_mask=attn_mask, logits_to_keep=1)
        last_logits = out.logits[:, -1, :]          # [B, V]
        # Custom CE: upcast only the slim [B, V] slice, not the full sequence.
        loss_full = torch.nn.functional.cross_entropy(last_logits.float(), target)
        loss = loss_full / GRAD_ACCUM_TRAIN
        loss.backward()
        running_loss += loss_full.item()
        # Drop intermediate tensors before next iter so the allocator can reuse.
        del out, last_logits, input_ids, attn_mask, target, loss, loss_full

        if (i + 1) % GRAD_ACCUM_TRAIN == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0,
            )
            optim.step(); sched.step(); optim.zero_grad()
            step += 1
            if step % 50 == 0:
                avg = running_loss / (GRAD_ACCUM_TRAIN * 50)
                running_loss = 0.0
                elapsed = (time.time()-t_start) / 60
                eta = elapsed / step * (total_steps - step)
                peak = torch.cuda.max_memory_allocated()/1024**3
                print(f"  step {step:>5}/{total_steps}  loss={avg:.4f}  "
                      f"lr={sched.get_last_lr()[0]:.2e}  elapsed={elapsed:.1f}min  "
                      f"eta={eta:.1f}min  peak_mem={peak:.1f}GB")
                # Periodic empty_cache reduces fragmentation between batches.
                torch.cuda.empty_cache()
            if step % CKPT_EVERY == 0:
                save_checkpoint(step)
                print(f"    [checkpoint] step {step} -> {LORA_DIR}")

print(f"\nTraining done in {(time.time()-t_start)/60:.1f} min")

# --- Final save (overwrites last periodic checkpoint with the actual end) --
save_checkpoint(step)
print(f"LoRA adapter -> {LORA_DIR}  (final step {step})")

# --- Aggressive cleanup so Phase 5 starts with a clean GPU ---------------
del model, optim, sched, dl, ds
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"\nGPU mem after Phase 4 cleanup:")
print(f"  allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"  reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")


## Phase 5 — Merge LoRA + re-score val_009 candidates (same HF path as Phase 3)

We merge the LoRA adapter into the base weights and re-run the same
`score_with_hf` function from Phase 3 on the cached `_baseline_prompts`. This
gives an apples-to-apples comparison: identical prompts, identical tokenizer,
identical scoring path — only the weights differ.


In [ ]:
# =============================================================================
# PHASE 5 — Merge LoRA + re-score val_009 candidates (HF transformers path)
# =============================================================================
# Uses the same score_with_hf function from Phase 3 (which uses
# logits_to_keep=1 — same memory-savings as during training).
# Apples-to-apples comparison: only the weights change between
# baseline_score and ft_score.

# --- Pre-flight cleanup ----------------------------------------------------
for _v in ("model", "scorer_model", "merged", "base"):
    if _v in globals():
        del globals()[_v]
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU mem at Phase 5 start: allocated={torch.cuda.memory_allocated()/1024**3:.2f} GB")

# --- Recovery: reload val_cands from disk if it's missing (kernel restart) -
if "val_cands" not in globals() or "baseline_score" not in val_cands.columns:
    print(f"[recovery] val_cands or baseline_score missing — reloading from disk")
    val_cands = pd.read_parquet(OUT_DIR / "val009_baseline_scores.parquet")
    assert "baseline_score" in val_cands.columns, \
        "Reloaded val_cands has no baseline_score column — Phase 3 must be redone"

# --- Recovery: rebuild _baseline_prompts if it's missing (kernel restart) --
if "_baseline_prompts" not in globals():
    print(f"[recovery] _baseline_prompts missing — rebuilding from val_cands")
    def _build_prompt(row):
        cand_fields = {
            "citation": row.citation, "code": row.code,
            "area": row.area or "", "role": row.role or "(unknown)",
            "title": row.title or "", "topic": row.topic or "",
            "concepts": row.concepts or "",
            "chapter_neighbors": row.chapter_neighbors or "(none)",
            "popularity_bucket": row.popularity_bucket,
            "text": row.text or "",
        }
        return format_full_prompt_for_training(query_text, query_meta, cand_fields)
    _baseline_prompts = [_build_prompt(r) for r in val_cands.itertuples()]
    print(f"  rebuilt {len(_baseline_prompts)} prompts")

# --- Merge LoRA into the base weights -------------------------------------
print(f"\nMerging LoRA from {LORA_DIR} into base ...")
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, str(LORA_DIR))
merged = merged.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
qwen3_tok.save_pretrained(str(MERGED_DIR))
print(f"Merged -> {MERGED_DIR}  in {time.time()-t0:.1f}s")

# Free CPU copy before moving merged to CUDA
del base
gc.collect()
merged = merged.to("cuda").eval()
print(f"GPU mem after move to CUDA: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# --- Re-score val_009 candidates with the fine-tuned model ---------------
print(f"\nFine-tuned scoring on the same {len(_baseline_prompts)} prompts ...")
t0 = time.time()
ft_scores = score_with_hf(merged, qwen3_tok, _baseline_prompts)
print(f"Done in {(time.time()-t0)/60:.1f} min")
val_cands["ft_score"] = ft_scores

# Persist ft_scores immediately (so Phase 6/7 work even after a kernel issue)
val_cands.to_parquet(OUT_DIR / "val009_scores_baseline_vs_ft.parquet", index=False)
print(f"[checkpoint] ft_scores saved to {OUT_DIR}/val009_scores_baseline_vs_ft.parquet")

gold_mean_ft   = val_cands[val_cands.is_gold].ft_score.mean()
non_mean_ft    = val_cands[~val_cands.is_gold].ft_score.mean()
gold_mean_base = val_cands[val_cands.is_gold].baseline_score.mean()
non_mean_base  = val_cands[~val_cands.is_gold].baseline_score.mean()

print(f"\n=== Score separation gold vs non-gold (0-1 probability) ===")
print(f"  Baseline:    gold={gold_mean_base:.4f}  non-gold={non_mean_base:.4f}  "
      f"sep={gold_mean_base-non_mean_base:+.4f}")
print(f"  Fine-tuned:  gold={gold_mean_ft:.4f}  non-gold={non_mean_ft:.4f}  "
      f"sep={gold_mean_ft-non_mean_ft:+.4f}")

del merged
for _ in range(3): gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"\nGPU mem after Phase 5 cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## Phase 6 — VERDICT: R@K side-by-side + per-gold rank + F1@K_gold

In [ ]:
# =============================================================================
# PHASE 6 — VERDICT (R@K, per-gold rank, F1@K_gold, decision)
# =============================================================================

def recall_at_k(df, score_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    return int(top.is_gold.sum()) / max(1, int(df.is_gold.sum()))

K_LIST     = [10, 25, 50, 100, 200, 500]
GOLD_TOTAL = 14   # full val_009 gold count (from gold_doc_sets)

print("="*80)
print(f"  RERANKER FINE-TUNE — val_009 (Stage A top-2000, 10 gold in pool)")
print("="*80)
print(f"\n{'K':<8}{'baseline R':>14}{'fine-tuned R':>14}{'delta':>10}{'rel %':>10}")
for K in K_LIST:
    base_r = recall_at_k(val_cands, "baseline_score", K)
    ft_r   = recall_at_k(val_cands, "ft_score",       K)
    delta  = ft_r - base_r
    rel    = (delta / max(1e-6, base_r)) * 100
    print(f"R@{K:<6}{base_r:>14.3f}{ft_r:>14.3f}{delta:>+10.3f}{rel:>+9.1f}%")

# R@K_gold = 14: the most relevant operating point for our task
base_rkg  = recall_at_k(val_cands, "baseline_score", GOLD_TOTAL)
ft_rkg    = recall_at_k(val_cands, "ft_score",       GOLD_TOTAL)
delta_rkg = ft_rkg - base_rkg
rel_rkg   = (delta_rkg / max(1e-6, base_rkg)) * 100
print(f"R@K_gold={GOLD_TOTAL:<3}{base_rkg:>14.3f}{ft_rkg:>14.3f}{delta_rkg:>+10.3f}{rel_rkg:>+9.1f}%")

# Per-gold rank comparison
print(f"\n=== Per-gold ranking (where each gold sits in the sorted list) ===")
val_sorted_base = val_cands.sort_values("baseline_score", ascending=False).reset_index(drop=True)
val_sorted_ft   = val_cands.sort_values("ft_score",       ascending=False).reset_index(drop=True)
print(f"  {'citation':<25}  {'baseline rank':>14}  {'fine-tuned rank':>16}  {'change':>10}")
gold_dids = set(val_cands[val_cands.is_gold].did)
for did in gold_dids:
    cit = val_cands[val_cands.did == did].iloc[0]["citation"]
    rank_base = int(val_sorted_base[val_sorted_base.did == did].index[0]) + 1
    rank_ft   = int(val_sorted_ft  [val_sorted_ft.did   == did].index[0]) + 1
    change = rank_base - rank_ft
    print(f"  {cit:<25}  {rank_base:>14}  {rank_ft:>16}  {change:>+10}")

# F1 at K = K_gold = 14
def _f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)
top_base = val_sorted_base.head(GOLD_TOTAL)
top_ft   = val_sorted_ft.head(GOLD_TOTAL)
correct_base = int(top_base.is_gold.sum())
correct_ft   = int(top_ft.is_gold.sum())
p_base = correct_base / GOLD_TOTAL ; r_base = correct_base / GOLD_TOTAL
p_ft   = correct_ft   / GOLD_TOTAL ; r_ft   = correct_ft   / GOLD_TOTAL
f1_base = _f1(p_base, r_base)
f1_ft   = _f1(p_ft,   r_ft)

print(f"\n=== F1 at K_gold ({GOLD_TOTAL}) ===")
print(f"  Baseline:    P={p_base:.3f}  R={r_base:.3f}  F1={f1_base:.3f}  "
      f"(correct={correct_base}/{GOLD_TOTAL})")
print(f"  Fine-tuned:  P={p_ft:.3f}  R={r_ft:.3f}  F1={f1_ft:.3f}  "
      f"(correct={correct_ft}/{GOLD_TOTAL})")
print(f"  Delta F1:    {f1_ft - f1_base:+.3f}")

print(f"\n" + "="*80)
print(f"  VERDICT")
print(f"="*80)
if rel_rkg >= 50.0:
    print(f"  CONCLUSIVE YES — R@{GOLD_TOTAL} relative lift {rel_rkg:+.1f}% >= 50%.")
    print(f"  Feature-enriched in-domain fine-tuning works on Swiss legal.")
    print(f"  Full-scale training pipeline (5M edges + synthetic queries) is justified.")
elif rel_rkg >= 20.0:
    print(f"  PROMISING — lift {rel_rkg:+.1f}% in [20%, 50%).")
    print(f"  Approach works but needs harder negatives + more diverse query synthesis.")
    print(f"  Worth scaling, but expect 0.45-0.55 F1 plateau without more work.")
else:
    print(f"  CONCLUSIVE NO — lift {rel_rkg:+.1f}% < 20%.")
    print(f"  Either: (a) training data doesn't transfer to val's question style,")
    print(f"          (b) the model can't learn from these features,")
    print(f"          (c) the eval is dominated by candidates the LLM cannot distinguish.")
    print(f"  Recommend: diagnose training loss curve + sample predictions before re-architecting.")


## Phase 7 — Save outputs

In [ ]:
# =============================================================================
# PHASE 7 — Save outputs (scores parquet + verdict.json)
# =============================================================================
val_cands.to_parquet(OUT_DIR / "val009_scores_baseline_vs_ft.parquet", index=False)
with open(OUT_DIR / "verdict.json", "w") as f:
    json.dump({
        "qid":             QID,
        "gold_total":      GOLD_TOTAL,
        "gold_in_topk":    int(val_cands.is_gold.sum()),
        "baseline":   {"R_at_K_gold": base_rkg, "F1_at_K_gold": f1_base,
                       "P": p_base, "correct": correct_base},
        "fine_tuned": {"R_at_K_gold": ft_rkg,   "F1_at_K_gold": f1_ft,
                       "P": p_ft,   "correct": correct_ft},
        "relative_lift_pct_R_at_K_gold": rel_rkg,
        "delta_F1": f1_ft - f1_base,
        "training_config": {
            "model": MODEL_NAME, "rank": LORA_RANK, "alpha": LORA_ALPHA,
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
            "lr": LR, "max_len_score": MAX_LEN_SCORE, "max_len_train": MAX_LEN_TRAIN,
        },
    }, f, indent=2)
print(f"Saved to {OUT_DIR}")
